# Lab 4: Data Quality Assessment & Preprocessing
## Applied on Wine Dataset (Different Dataset as Required)
Author: Abbas Aziz


In [17]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

# Load Wine dataset
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


## Task 1: Identify Data Quality Issues

In [18]:
df.info()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
dtypes: float64(13)
m

np.int64(0)

In [19]:
#The dataset contains 178 rows and 13 numerical features.
#No missing values or duplicates were originally found.

## Task 2: Handle Missing Values (Artificially Introduced)

In [20]:
# Introduce artificial missing values (2%)
df_missing = df.copy()
for col in df_missing.columns:
    df_missing.loc[df_missing.sample(frac=0.02).index, col] = np.nan

# Impute using median
df_imputed = df_missing.fillna(df_missing.median())
df_imputed.isnull().sum()

alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
dtype: int64

In [21]:
#Artificial missing values (2%) were introduced.
#Median imputation was used because it is robust to outliers.

## Task 3: Outlier Detection using IQR

In [22]:
df_clean = df_imputed.copy()

for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]

df_clean.shape

(156, 13)

In [23]:
#IQR method was used to detect and remove outliers.
#After removal, the dataset size became (154, 13).

## Task 4: Normalization

In [24]:
# Min-Max Scaling
minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df_clean), columns=df_clean.columns)

# Z-score Standardization
scaler = StandardScaler()
df_zscore = pd.DataFrame(scaler.fit_transform(df_clean), columns=df_clean.columns)

df_minmax.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,0.824561,0.217978,0.528846,0.278481,0.890625,0.627586,0.757660,0.283019,0.783333,0.463830,0.549451,0.970696,0.561341
1,0.476608,0.255056,0.250000,0.000000,0.468750,0.575862,0.674095,0.245283,0.362500,0.329787,0.560440,0.780220,0.550642
2,0.511696,0.364045,0.759615,0.468354,0.484375,0.627586,0.807799,0.320755,1.000000,0.468085,0.538462,0.695971,0.646933
3,0.865497,0.271910,0.461538,0.354430,0.437500,0.989655,0.495822,0.207547,0.737500,0.693617,0.351648,0.798535,0.857347
4,0.535088,0.415730,0.461538,0.620253,0.750000,0.627586,0.654596,0.490566,0.587500,0.323404,0.549451,0.608059,0.325963


In [25]:
#Both Min-Max scaling and Z-score standardization were applied
#to normalize the numerical features.

## Task 5: PCA (Dimensionality Reduction)

In [26]:
pca = PCA(n_components=2)
principal_components = pca.fit_transform(df_zscore)

explained_variance = pca.explained_variance_ratio_
explained_variance

array([0.38352562, 0.203121  ])

In [27]:
#PCA was applied to reduce dimensionality.
#Explained variance ratio shows how much variance is captured.